In [1]:
import functools

import numpy as np
import mediapy as media
import mujoco
import jax
import jax.numpy as jp
from mujoco_playground import registry
from pathlib import Path
from brax.training.agents.ppo import checkpoint

In [2]:
env_name = "G1JoystickFlatTerrain"
policy_path = "../checkpoints/ppo_g1/g1_walking_policy_v3/000000000001"

In [3]:
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)
env_cfg.push_config.enable = False  # Disable random pushes

In [4]:
# Load policy from checkpoint
checkpoint_path = str(Path(policy_path).absolute())
custom_policy_fn = checkpoint.load_policy(checkpoint_path)

In [7]:
# Rollout and Render policy
from mujoco_playground._src.gait import draw_joystick_command

env = registry.load(env_name)
eval_env = registry.load(env_name)
jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_inference_fn = jax.jit(custom_policy_fn)

rng = jax.random.PRNGKey(1)

rollout = []
modify_scene_fns = []

x_vel = 0.5  #@param {type: "number"}
y_vel = 0.5  #@param {type: "number"}
yaw_vel = 0.0  #@param {type: "number"}
command = jp.array([x_vel, y_vel, yaw_vel])

phase_dt = 2 * jp.pi * eval_env.dt * 1.5
phase = jp.array([0, jp.pi])

for j in range(1):
  print(f"episode {j}")
  state = jit_reset(rng)
  state.info["phase_dt"] = phase_dt
  state.info["phase"] = phase
  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    if state.done:
      break
    state.info["command"] = command
    rollout.append(state)

    xyz = np.array(state.data.xpos[eval_env.mj_model.body("torso_link").id])
    xyz += np.array([0, 0.0, 0])
    x_axis = state.data.xmat[eval_env._torso_body_id, 0]
    yaw = -np.arctan2(x_axis[1], x_axis[0])
    modify_scene_fns.append(
        functools.partial(
            draw_joystick_command,
            cmd=state.info["command"],
            xyz=xyz,
            theta=yaw,
            scl=np.linalg.norm(state.info["command"]),
        )
    )

render_every = 1
fps = 1.0 / eval_env.dt / render_every
print(f"fps: {fps}")
traj = rollout[::render_every]
mod_fns = modify_scene_fns[::render_every]

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = False

frames = eval_env.render(
    traj,
    camera="track",
    scene_option=scene_option,
    width=640*2,
    height=480,
    modify_scene_fns=mod_fns,
)
media.show_video(frames, fps=fps, loop=False)

episode 0
fps: 50.0


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 344/344 [00:08<00:00, 39.59it/s]


This browser does not support the video tag.


## Observation Space

In [ ]:
# create state
state = jit_reset(rng)
state.info["phase_dt"] = phase_dt
state.info["phase"] = phase

In [ ]:
# State.obs is a dict containing
#   state            -> (103, ) useful for training policy
#   privileged_state -> (216, ) info available in the simulation
print(f"obs keys: {state.obs.keys()}\n")
for k, v in state.obs.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")

If you run this you can inspect the source to see how state is constructed:

```python
import inspect
print(inspect.getsource(eval_env.unwrapped.__class__))
```

103 dims of the state vector:

| Component   | Dims | Notes |
| ----------- | ---- | ----- |
| noisy_linvel|   3  | base linear velocity |
| noisy_gyro  |   3  | base angular velocity |
| noisy_gravity|   3  | projected gravity vector |
| command     |   3  | [1, 0, 0] in our case |
| noisy_joint_angles - default_pose | 29 | relative joint positions |
| last_act | 29 | previous action |
| phase | 2 | sin/cos of gait phase |

In [ ]:
import inspect
print(inspect.getsource(eval_env.unwrapped.__class__))

## Action space

In [ ]:
act_rng, rng = jax.random.split(rng)

act_rng, rng = jax.random.split(rng)
ctrl, _ = jit_inference_fn(state.obs, act_rng)
print("ctrl shape:", ctrl.shape) # 29 dim one for each joint
print("ctrl example:", ctrl)

In [11]:
print(env_cfg.episode_length)
print(eval_env.dt)
print(env_cfg.episode_length * eval_env.dt, "seconds")

1000
0.02
20.0 seconds


In [12]:
# Data collection
import os
import numpy as np

n_episodes = 200
save_path = Path("../data/demonstrations/g1_walking/")

all_episodes = []

In [13]:
env = registry.load(env_name)
eval_env = registry.load(env_name)
jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_inference_fn = jax.jit(custom_policy_fn)

rng = jax.random.PRNGKey(1)

x_vel = 1.0
y_vel = 0.0
yaw_vel = 0.0 
command = jp.array([x_vel, y_vel, yaw_vel])

phase_dt = 2 * jp.pi * eval_env.dt * 1.5
phase = jp.array([0, jp.pi])

for j in range(n_episodes):
    print(f"episode {j}")
    state = jit_reset(rng)
    state.info["phase_dt"] = phase_dt
    state.info["phase"] = phase

    episode = {"obs": [], "action": []}
    
    for i in range(env_cfg.episode_length):
        act_rng, rng = jax.random.split(rng)

        obs = np.array(state.obs["state"]) # (103,)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        action = np.array(ctrl) # (29,)

        episode["obs"].append(obs)
        episode["action"].append(action)

        state = jit_step(state, ctrl)
        if state.done:
          break
        state.info["command"] = command

    episode["obs"] = np.array(episode["obs"])
    episode["action"] = np.array(episode["action"])
    all_episodes.append(episode)


episode 0
episode 1
episode 2
episode 3
episode 4
episode 5
episode 6
episode 7
episode 8
episode 9
episode 10
episode 11
episode 12
episode 13
episode 14
episode 15
episode 16
episode 17
episode 18
episode 19
episode 20
episode 21
episode 22
episode 23
episode 24
episode 25
episode 26
episode 27
episode 28
episode 29
episode 30
episode 31
episode 32
episode 33
episode 34
episode 35
episode 36
episode 37
episode 38
episode 39
episode 40
episode 41
episode 42
episode 43
episode 44
episode 45
episode 46
episode 47
episode 48
episode 49
episode 50
episode 51
episode 52
episode 53
episode 54
episode 55
episode 56
episode 57
episode 58
episode 59
episode 60
episode 61
episode 62
episode 63
episode 64
episode 65
episode 66
episode 67
episode 68
episode 69
episode 70
episode 71
episode 72
episode 73
episode 74
episode 75
episode 76
episode 77
episode 78
episode 79
episode 80
episode 81
episode 82
episode 83
episode 84
episode 85
episode 86
episode 87
episode 88
episode 89
episode 90
episode 9

In [19]:
min_length = 1000
filtered_episodes = [ep for ep in all_episodes if len(ep["obs"]) >= min_length]

In [22]:
Counter([len(ep["obs"]) for ep in filtered_episodes])

Counter({1000: 158})

In [24]:
np.save(save_path / "demos.npy", filtered_episodes)

In [27]:
np.save(save_path / "meta.npy", {
    "obs_dim": 103,
    "action_dim": 29,
    "dt": 0.02,
    "hz": 50,
    "command": [1.0, 0.0, 0.0],
    "n_episodes": len(filtered_episodes),
})

In [28]:
demos = np.load(save_path / "demos.npy", allow_pickle=True)
print(len(demos))

158


In [30]:
print(type(demos[0]))
print(demos[0].keys())
print(demos[0]["obs"].shape)
print(demos[0]["action"].shape)

<class 'dict'>
dict_keys(['obs', 'action'])
(1000, 103)
(1000, 29)
